In [ ]:
from pathlib import Path\nimport math\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom IPython.display import display\n\nfrom study_analysis import (\n    AGGREGATE_STATS_FILENAME,\n    PAIRWISE_STATS_FILENAME,\n    RUN_SUMMARY_FILENAME,\n    build_round_history,\n    load_latest_study_dir,\n    refresh_study_outputs,\n)\n\n# Set STUDY_DIR to a specific folder if you do not want to use study_runs/latest_study.txt.\nSTUDY_DIR = None\nOUTPUT_ROOT = Path('study_runs')\n\nstudy_dir = Path(STUDY_DIR) if STUDY_DIR else load_latest_study_dir(OUTPUT_ROOT)\nrun_summary_df, aggregate_df, pairwise_df = refresh_study_outputs(study_dir)\nround_history_df = build_round_history(study_dir)\n\nprint(f'Study directory: {study_dir}')\nprint(f'Completed runs: {len(run_summary_df)}')\nprint(f'Round-history rows: {len(round_history_df)}')\nprint(f'Summary CSV: {study_dir / RUN_SUMMARY_FILENAME}')\nprint(f'Aggregate CSV: {study_dir / AGGREGATE_STATS_FILENAME}')\nprint(f'Pairwise CSV: {study_dir / PAIRWISE_STATS_FILENAME}')\n\nif run_summary_df.empty:\n    raise RuntimeError('No completed runs found in the selected study.')\nif round_history_df.empty:\n    raise RuntimeError('Round history is empty. Check archived CSV paths in the manifest.')\n\ndisplay(run_summary_df.sort_values(['dataset', 'model', 'learning_type_display', 'repetition']).reset_index(drop=True))\ndisplay(aggregate_df.sort_values(['dataset', 'model', 'learning_type_display', 'metric']).reset_index(drop=True))\nif not pairwise_df.empty:\n    display(pairwise_df.sort_values(['dataset', 'model', 'metric', 'approach_a', 'approach_b']).reset_index(drop=True))\n\nplot_metrics = [\n    'Accuracy',\n    'Duration_Sec',\n    'TFLOPS',\n    'Round_Throughput_TFLOPS',\n    'Avg_Training_Time_Sec',\n    'Avg_Communication_Time_Sec',\n]\navailable_metrics = [metric for metric in plot_metrics if metric in round_history_df.columns]\nfor metric in available_metrics:\n    round_history_df[metric] = pd.to_numeric(round_history_df[metric], errors='coerce')\n\ngroup_keys = [\n    'dataset',\n    'model',\n    'effective_num_clients',\n    'effective_clients_per_round',\n    'rounds',\n    'epochs',\n    'data_distr',\n    'comparison_profile',\n]\n\nfor group_values, group_df in round_history_df.groupby(group_keys, dropna=False):\n    if not isinstance(group_values, tuple):\n        group_values = (group_values,)\n    group_payload = dict(zip(group_keys, group_values))\n    title_prefix = (\n        f"{str(group_payload['dataset']).upper()} | {str(group_payload['model']).upper()} | "\n        f"N={group_payload['effective_num_clients']} | k={group_payload['effective_clients_per_round']} | "\n        f"rounds={group_payload['rounds']} | epochs={group_payload['epochs']} | alpha={group_payload['data_distr']}"\n    )\n    approaches = list(group_df['learning_type_display'].dropna().unique())\n    ncols = 2\n    nrows = math.ceil(len(available_metrics) / ncols)\n    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4.5 * nrows))\n    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]\n\n    for axis, metric in zip(axes, available_metrics):\n        metric_df = group_df[['learning_type_display', 'Round', metric]].copy()\n        metric_df[metric] = pd.to_numeric(metric_df[metric], errors='coerce')\n        metric_df = metric_df.dropna()\n        if metric_df.empty:\n            axis.set_visible(False)\n            continue\n        aggregated = (\n            metric_df.groupby(['learning_type_display', 'Round'], dropna=False)[metric]\n            .agg(['mean', 'std', 'count'])\n            .reset_index()\n        )\n        for approach in approaches:\n            approach_df = aggregated[aggregated['learning_type_display'] == approach].sort_values('Round')\n            if approach_df.empty:\n                continue\n            rounds = approach_df['Round'].to_numpy(dtype=float)\n            means = approach_df['mean'].to_numpy(dtype=float)\n            stds = approach_df['std'].fillna(0.0).to_numpy(dtype=float)\n            axis.plot(rounds, means, marker='o', label=approach)\n            axis.fill_between(rounds, means - stds, means + stds, alpha=0.2)\n        axis.set_title(metric)\n        axis.set_xlabel('Round')\n        axis.set_ylabel(metric)\n        axis.grid(True, alpha=0.25)\n        axis.legend()\n\n    for axis in axes[len(available_metrics):]:\n        axis.set_visible(False)\n\n    fig.suptitle(title_prefix, fontsize=14)\n    fig.tight_layout(rect=[0, 0, 1, 0.97])\n    plt.show()\n\nsummary_metrics = [\n    metric\n    for metric in [\n        'Final_Accuracy',\n        'Best_Accuracy',\n        'Mean_Accuracy',\n        'Mean_Duration_Sec',\n        'Mean_Avg_Training_Time_Sec',\n        'Mean_Avg_Communication_Time_Sec',\n    ]\n    if not aggregate_df.empty and metric in aggregate_df['metric'].unique()\n]\n\nfor metric in summary_metrics:\n    metric_df = aggregate_df[aggregate_df['metric'] == metric].copy()\n    if metric_df.empty:\n        continue\n    fig, axis = plt.subplots(figsize=(12, 5))\n    metric_df = metric_df.sort_values(['dataset', 'model', 'learning_type_display'])\n    labels = [\n        f"{row.dataset.upper()} | {row.model.upper()} | {row.learning_type_display}"\n        for row in metric_df.itertuples()\n    ]\n    means = metric_df['mean'].to_numpy(dtype=float)\n    stds = metric_df['std'].fillna(0.0).to_numpy(dtype=float)\n    axis.bar(range(len(labels)), means, yerr=stds, capsize=4)\n    axis.set_xticks(range(len(labels)))\n    axis.set_xticklabels(labels, rotation=45, ha='right')\n    axis.set_title(f'Mean across repetitions: {metric}')\n    axis.set_ylabel(metric)\n    axis.grid(True, axis='y', alpha=0.25)\n    fig.tight_layout()\n    plt.show()\n